# Mask Models

Once we have the data, we can build some models to test them and see how they perform versus one another. Something we care about is the performance of the model (accuracy) of predicting whether or not something is a bulk layer or not.


In [3]:
%pip install PyWavelets
%pip install tqdm
%pip install autogluon

In [ ]:
import os

import numpy as np
import pandas as pd
from tqdm import tqdm
import pywt
import autogluon.core as ag
from autogluon.tabular import TabularDataset, TabularPredictor

In [4]:
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor
from sklearn.model_selection import train_test_split

# ── 1. Load your preprocessed CSV ──────────────────────────────────────────
feature_function_name = "wavelet_features"  # or "simple_statistics"
resolution = 4                             # whichever resolution you used

df = pd.read_csv(f"./new_csvs/{feature_function_name}_resolution_{resolution}.csv")

# ── 2. Ensure target is binary ─────────────────────────────────────────────
# Inspect unique values first
print("Unique 'bulk' values:", df['bulk'].unique())

# If bulk is numeric, binarize it (adjust threshold as needed)
# e.g. if 0 = no error, anything else = error
df['target'] = (df['bulk'] != 0).astype(int)

# ── 3. Drop columns that would leak identity info ──────────────────────────
# 'file' and 'bulk' are not features — drop them
# Keep: scan_number, subsegment_index, mean_laser_current, all wavelet/stat cols
drop_cols = ['file', 'bulk']
df = df.drop(columns=drop_cols)

print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['target'].value_counts()}")

# ── 4. Train/test split — split by file to avoid data leakage ──────────────
# (if you kept 'file' col, group-split here instead)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

# ── 5. Convert to AutoGluon Dataset ───────────────────────────────────────
train_data = TabularDataset(train_df)
test_data  = TabularDataset(test_df)

# ── 6. Train ───────────────────────────────────────────────────────────────
predictor = TabularPredictor(
    label='target',
    problem_type='binary',
    eval_metric='roc_auc',      # good default for imbalanced binary classification
    path='./autogluon_models'   # saves models here for reuse
).fit(
    train_data,
    time_limit=600,             # seconds — increase for better results
    presets='best_quality',     # or 'medium_quality' for faster runs
    verbosity=2
)

# ── 7. Evaluate ────────────────────────────────────────────────────────────
leaderboard = predictor.leaderboard(test_data, silent=False)
print(leaderboard)

# ── 8. Detailed performance metrics ───────────────────────────────────────
performance = predictor.evaluate(test_data)
print("\nPerformance metrics:", performance)

# ── 9. Feature importance ──────────────────────────────────────────────────
importance = predictor.feature_importance(test_data)
print("\nFeature importance:\n", importance)

Unique 'bulk' values: [1. 0.]
Dataset shape: (60914, 10)
Target distribution:
target
1    49647
0    11267
Name: count, dtype: int64


Verbosity: 2 (Standard Logging)


Train: (48731, 10), Test: (12183, 10)


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       10.78 GB / 12.67 GB (85.1%)
Disk Space Avail:   74.68 GB / 107.72 GB (69.3%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will 

                     model  score_test  score_val eval_metric  pred_time_test  pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0        LightGBMXT_BAG_L1    0.892236   0.882470     roc_auc        5.440116       2.590427   54.565732                 5.440116                2.590427          54.565732            1       True          1
1      WeightedEnsemble_L2    0.892013   0.883686     roc_auc        8.458938       7.593300  188.221246                 0.003692                0.010242           0.786920            2       True          4
2      WeightedEnsemble_L3    0.891847   0.884051     roc_auc       10.377914       9.751676  276.247852                 0.004281                0.015033           1.902513            3       True          7
3        LightGBMXT_BAG_L2    0.891208   0.883480     roc_auc        9.750644       9.062785  228.822569                 1.295397                1.479726          41.38

Computing feature importance via permutation shuffling for 9 features using 5000 rows with 5 shuffle sets...



Performance metrics: {'roc_auc': np.float64(0.8918468784375673), 'accuracy': 0.9061807436591972, 'balanced_accuracy': np.float64(0.7796235432313814), 'mcc': np.float64(0.6611740203146637), 'f1': 0.9445549357264128, 'precision': 0.9111839026672905, 'recall': 0.9804632426988923}


	192.24s	= Expected runtime (38.45s per shuffle set)
	183.28s	= Actual runtime (Completed 5 of 5 shuffle sets)



Feature importance:
                               importance    stddev       p_value  n  p99_high  \
wavelet_level_4_energy_ratio    0.146795  0.007002  6.191602e-07  5  0.161211   
mean_laser_current              0.053504  0.002679  7.517682e-07  5  0.059021   
scan_number                     0.026821  0.003369  2.926297e-05  5  0.033758   
subsegment_index                0.018419  0.001379  3.738489e-06  5  0.021258   
wavelet_level_5_energy_ratio    0.017109  0.003137  1.297427e-04  5  0.023568   
wavelet_level_3_energy_ratio    0.009534  0.001701  1.167011e-04  5  0.013037   
wavelet_level_0_energy_ratio    0.001750  0.001307  2.008189e-02  5  0.004442   
wavelet_level_2_energy_ratio    0.001652  0.001204  1.866566e-02  5  0.004131   
wavelet_level_1_energy_ratio    0.000122  0.000745  3.660883e-01  5  0.001656   

                               p99_low  
wavelet_level_4_energy_ratio  0.132378  
mean_laser_current            0.047988  
scan_number                   0.019883  
sub

In [7]:
from sklearn.model_selection import train_test_split

# ── 1. Load your preprocessed CSV ──────────────────────────────────────────
feature_function_name = "wavelet_features"
resolution = 4

df = pd.read_csv(f"./new_csvs/{feature_function_name}_resolution_{resolution}.csv")
df['target'] = (df['bulk'] != 0).astype(int)
df = df.drop(columns=['bulk'])

# ── 2. Define masking function ─────────────────────────────────────────────
def mask_layer(df, fraction):
    """
    Keep only the first `fraction` of subsegments per (file, scan_number).
    e.g. fraction=0.2 keeps the first 20% of subsegments in each scan.
    """
    def keep_first_fraction(group):
        n_keep = max(1, int(np.ceil(len(group) * fraction)))
        return group.iloc[:n_keep]

    return (
        df.groupby(['file', 'scan_number'], group_keys=False)
          .apply(keep_first_fraction)
          .reset_index(drop=True)
    )

# ── 3. Run AutoGluon for each mask fraction ────────────────────────────────
fractions = [0.2, 0.3, 0.5, 0.7]
results = {}  # stores performance metrics per fraction

for fraction in fractions:
    print(f"\n{'='*60}")
    print(f"  Training on first {int(fraction*100)}% of each layer")
    print(f"{'='*60}")

    # Apply mask
    masked_df = mask_layer(df, fraction)
    masked_df = masked_df.drop(columns=['file'])  # drop after masking

    print(f"  Masked dataset shape: {masked_df.shape}")
    print(f"  Target distribution:\n{masked_df['target'].value_counts()}")

    # Train/test split
    train_df, test_df = train_test_split(
        masked_df,
        test_size=0.2,
        random_state=42,
        stratify=masked_df['target']
    )

    train_data = TabularDataset(train_df)
    test_data  = TabularDataset(test_df)

    # Train
    predictor = TabularPredictor(
        label='target',
        problem_type='binary',
        eval_metric='roc_auc',
        path=f'./autogluon_models/fraction_{int(fraction*100)}pct'
    ).fit(
        train_data,
        time_limit=600,
        presets='best_quality',
        verbosity=1
    )

    # Evaluate
    performance = predictor.evaluate(test_data)
    leaderboard = predictor.leaderboard(test_data, silent=True)
    best_model  = leaderboard.iloc[0]  # top row = best model

    results[fraction] = {
        'performance': performance,
        'best_model_name': best_model['model'],
        'best_model_score': best_model['score_test'],
        'leaderboard': leaderboard
    }

    print(f"\n  ✅ Best model: {best_model['model']} | ROC-AUC: {best_model['score_test']:.4f}")

# ── 4. Summary table ───────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  SUMMARY")
print(f"{'='*60}")

summary = pd.DataFrame([
    {
        'fraction': f"{int(f*100)}%",
        'best_model': results[f]['best_model_name'],
        'roc_auc': results[f]['best_model_score'],
        **results[f]['performance']  # expands all metrics (accuracy, f1, etc.)
    }
    for f in fractions
])

print(summary.to_string(index=False))
summary.to_csv('./autogluon_models/summary.csv', index=False)
print("\nSummary saved to ./autogluon_models/summary.csv")


  Training on first 20% of each layer


/tmp/ipykernel_566/4211773679.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(keep_first_fraction)


  Masked dataset shape: (35443, 10)
  Target distribution:
target
1    26823
0     8620
Name: count, dtype: int64


	Not enough time to generate out-of-fold predictions for model. Estimated time required was 10.25s compared to 10s of available time.



  ✅ Best model: WeightedEnsemble_L3 | ROC-AUC: 0.9054

  Training on first 30% of each layer


...

  Masked dataset shape: (40052, 10)
  Target distribution:
target
1    30874
0     9178
Name: count, dtype: int64

  ✅ Best model: LightGBMXT_BAG_L2 | ROC-AUC: 0.9029

  Training on first 70% of each layer


/tmp/ipykernel_566/4211773679.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(keep_first_fraction)


  Masked dataset shape: (60713, 10)
  Target distribution:
target
1    49486
0    11227
Name: count, dtype: int64


	Not enough time to generate out-of-fold predictions for model. Estimated time required was 13.68s compared to 10s of available time.



  ✅ Best model: LightGBMXT_BAG_L1 | ROC-AUC: 0.8793

  SUMMARY
fraction          best_model  roc_auc  accuracy  balanced_accuracy      mcc       f1  precision   recall
     20% WeightedEnsemble_L3 0.905442  0.898857           0.824333 0.711785 0.935516   0.903893 0.969432
     30% WeightedEnsemble_L3 0.890240  0.892972           0.812669 0.693504 0.932002   0.897837 0.968872
     50%   LightGBMXT_BAG_L2 0.902393  0.900262           0.812642 0.701474 0.937739   0.903725 0.974413
     70%   LightGBMXT_BAG_L1 0.878885  0.903483           0.773416 0.650208 0.943018   0.908903 0.979794

Summary saved to ./autogluon_models/summary.csv


We're gonna be so honest here. Jonathan's scared of installing autogluon and destroying his computer and he did it last year, so he did this on Google Collab.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import shutil
shutil.make_archive(
    base_name='/content/drive/MyDrive/autogluon_models',  # destination
    format='zip',
    root_dir='/content',
    base_dir='autogluon_models'                           # folder to zip
)

'/content/drive/MyDrive/autogluon_models.zip'